<a href="https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%pip install -q pandas numpy scikit-learn matplotlib

In [12]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kaifahmad236/flyrank-ml-internship.git"
REPO_DIR = Path("/content/flyrank-ml-internship")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)

print("Repository:", REPO_DIR)
print("Current directory:", os.getcwd())

Repository: /content/flyrank-ml-internship
Current directory: /content/flyrank-ml-internship


In [13]:
RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Raw dataset exists:", RAW_PATH.exists())

if not RAW_PATH.exists():
    raise FileNotFoundError(
        "data/raw/content_refresh_anonymized.csv was not found "
        "in the repository."
    )

print("Dataset path:", RAW_PATH)

Raw dataset exists: True
Dataset path: data/raw/content_refresh_anonymized.csv


In [14]:
subprocess.run(
    [
        sys.executable,
        "scripts/01_prepare_features.py"
    ],
    check=True
)

print("Feature preparation completed.")

Feature preparation completed.


In [15]:
subprocess.run(
    [
        sys.executable,
        "scripts/02_baseline_score.py"
    ],
    check=True
)

print("Baseline preparation completed.")

Baseline preparation completed.


In [16]:
import pandas as pd
import numpy as np

FEATURE_PATH = Path(
    "data/processed/refresh_feature_vector.csv"
)

BASELINE_PATH = Path(
    "data/processed/baseline_refresh_queue.csv"
)

feature_df = pd.read_csv(FEATURE_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

print("Feature rows:", len(feature_df))
print("Feature columns:", len(feature_df.columns))
print("Baseline rows:", len(baseline_df))

display(feature_df.head(3))

Feature rows: 30000
Feature columns: 52
Baseline rows: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Random Forest for this modeling step.

The purpose of the model is to identify pages that are more likely to belong to the observed declining group and to compare that ranking with the Week-4 rule-based baseline. Random Forest is suitable because it can capture non-linear relationships between search, traffic, content, and engagement signals without requiring feature scaling.

I chose this method because the goal is not to use the most complicated model. The important question is whether a learned model provides better decision support than the simpler baseline. Random Forest also provides feature importance, which makes the model easier to inspect and explain.

The model uses observable features only. The target and label-derived fields are kept out of the feature matrix to avoid leakage.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

print("Selected model: Random Forest")
print("Random state:", RANDOM_STATE)

Selected model: Random Forest
Random state: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped holdout rather than randomly splitting individual pages.

Pages belonging to the same client can share similar search and content patterns. If pages from one client appeared in both training and testing data, the test result could be overly optimistic because the model would already have seen related examples from that client.

I therefore hold out approximately 20% of clients for testing and keep all pages from each client on only one side of the split.

The split uses a fixed random seed of 42 so that the experiment is reproducible. The Week-4 baseline and the Random Forest are evaluated on the exact same held-out clients.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split

all_indices = np.arange(len(feature_df))

client_series = (
    feature_df["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = (
    client_series
    .drop_duplicates()
    .to_numpy()
)

random_generator = np.random.default_rng(
    RANDOM_STATE
)

shuffled_clients = random_generator.permutation(
    unique_clients
)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

test_mask = client_series.isin(
    test_clients
).to_numpy()

train_indices = all_indices[
    ~test_mask
]

test_indices = all_indices[
    test_mask
]

target_series = (
    feature_df["is_declining_label"]
    .astype(int)
)

# Verify that both splits contain both target classes.
if (
    target_series.iloc[train_indices].nunique() == 2
    and
    target_series.iloc[test_indices].nunique() == 2
):

    split_strategy = "client_holdout"

else:

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=target_series
    )

    split_strategy = "stratified_row_holdout"


train_df = feature_df.iloc[
    train_indices
].copy()

test_df = feature_df.iloc[
    test_indices
].copy()

train_clients = set(
    train_df["client_id"]
    .fillna("unknown")
    .astype(str)
)

test_clients = set(
    test_df["client_id"]
    .fillna("unknown")
    .astype(str)
)

client_overlap = (
    train_clients.intersection(
        test_clients
    )
)

print("Split strategy:", split_strategy)
print("Total rows:", len(feature_df))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0

print("Grouped validation check passed.")

Split strategy: client_holdout
Total rows: 30000
Training rows: 27675
Test rows: 2325
Training clients: 26
Test clients: 6
Client overlap: 0
Grouped validation check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The target is `is_declining_label`, defined from the observed trend direction. The target is used only as the outcome and is not included as a model feature.

I use the same held-out test clients for both the Random Forest and the Week-4 baseline. Precision@50 is the main comparison metric because the practical decision is to prioritize a small number of pages for review.

I also report accuracy, precision, recall, F1, ROC AUC, and Average Precision for the Random Forest. The model is considered useful only if its results provide meaningful improvement over the simpler baseline rather than simply because it is more complex.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


# ------------------------------------------------------------
# Build the feature matrix
# ------------------------------------------------------------

# Use the feature lists defined by the repository's ML utilities.
sys.path.insert(
    0,
    str(Path("scripts").resolve())
)

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES
)


numeric_features = [
    column
    for column in MODEL_NUMERIC_FEATURES
    if column in feature_df.columns
]

categorical_features = [
    column
    for column in MODEL_CATEGORICAL_FEATURES
    if column in feature_df.columns
]


numeric_frame = (
    feature_df[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
)

numeric_frame = (
    numeric_frame
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


categorical_frame = (
    feature_df[categorical_features]
    .fillna("unknown")
    .astype(str)
)


encoded_frame = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float
)


feature_matrix = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True)
    ],
    axis=1
)


feature_columns = list(
    feature_matrix.columns
)


print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Final model features:", len(feature_columns))


# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

target = (
    feature_df["is_declining_label"]
    .astype(int)
)


# ------------------------------------------------------------
# Train/test data
# ------------------------------------------------------------

X_train = feature_matrix.iloc[
    train_indices
]

X_test = feature_matrix.iloc[
    test_indices
]

y_train = target.iloc[
    train_indices
]

y_test = target.iloc[
    test_indices
]


# ------------------------------------------------------------
# Train Random Forest
# ------------------------------------------------------------

model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# Test probabilities
# ------------------------------------------------------------

model_probability = model.predict_proba(
    X_test
)[:, 1]

model_prediction = (
    model_probability >= 0.5
).astype(int)


# ------------------------------------------------------------
# Precision@K
# ------------------------------------------------------------

def precision_at_k(
    target_values,
    scores,
    k
):

    target_values = np.asarray(
        target_values
    )

    scores = np.asarray(
        scores
    )

    k = min(
        k,
        len(target_values)
    )

    order = np.argsort(
        -scores,
        kind="mergesort"
    )[:k]

    return float(
        target_values[order].mean()
    )


# ------------------------------------------------------------
# Random Forest metrics
# ------------------------------------------------------------

model_accuracy = accuracy_score(
    y_test,
    model_prediction
)

model_precision = precision_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_recall = recall_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_f1 = f1_score(
    y_test,
    model_prediction,
    zero_division=0
)

model_precision_20 = precision_at_k(
    y_test,
    model_probability,
    20
)

model_precision_50 = precision_at_k(
    y_test,
    model_probability,
    50
)

model_precision_100 = precision_at_k(
    y_test,
    model_probability,
    100
)

model_auc = roc_auc_score(
    y_test,
    model_probability
)

model_average_precision = (
    average_precision_score(
        y_test,
        model_probability
    )
)


# ------------------------------------------------------------
# Week-4 baseline
# ------------------------------------------------------------

baseline_lookup = (
    baseline_df
    .set_index("content_id")
    ["baseline_refresh_score"]
)

baseline_test_scores = (
    feature_df.iloc[
        test_indices
    ]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)


baseline_precision_20 = precision_at_k(
    y_test,
    baseline_test_scores,
    20
)

baseline_precision_50 = precision_at_k(
    y_test,
    baseline_test_scores,
    50
)

baseline_precision_100 = precision_at_k(
    y_test,
    baseline_test_scores,
    100
)


# ------------------------------------------------------------
# Model vs baseline table
# ------------------------------------------------------------

comparison = pd.DataFrame(
    {
        "Method": [
            "Week-4 baseline",
            "Random Forest"
        ],

        "Precision@20": [
            baseline_precision_20,
            model_precision_20
        ],

        "Precision@50": [
            baseline_precision_50,
            model_precision_50
        ],

        "Precision@100": [
            baseline_precision_100,
            model_precision_100
        ],

        "Accuracy": [
            np.nan,
            model_accuracy
        ],

        "Precision": [
            np.nan,
            model_precision
        ],

        "Recall": [
            np.nan,
            model_recall
        ],

        "F1": [
            np.nan,
            model_f1
        ],

        "ROC AUC": [
            np.nan,
            model_auc
        ],

        "Average Precision": [
            np.nan,
            model_average_precision
        ]
    }
)


print("Model vs baseline:")
display(comparison)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "importance": model.feature_importances_
        }
    )
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top model features:")
display(
    feature_importance.head(15)
)


# ------------------------------------------------------------
# Save test results for Section 4
# ------------------------------------------------------------

results = feature_df.iloc[
    test_indices
][
    [
        "content_id",
        "client_id",
        "is_declining_label"
    ]
].copy()


results["model_probability"] = (
    model_probability
)

results["model_prediction"] = (
    model_prediction
)

results["baseline_score"] = (
    baseline_test_scores
)


print("Section 3 completed.")

Numeric features: 18
Categorical features: 8
Final model features: 52
Model vs baseline:


,Method,Precision@20,Precision@50,Precision@100,Accuracy,Precision,Recall,F1,ROC AUC,Average Precision
0,Week-4 baseline,0.15,0.24,0.36,NaN,NaN,NaN,NaN,NaN,NaN
1,Random Forest,0.65,0.74,0.72,0.672258,0.560996,0.743674,0.639546,0.75003,0.618219


Top model features:


,feature,importance
0,days_with_impressions,0.134951
1,log_impressions_90d,0.129377
2,avg_position,0.109203
3,content_age_days,0.092048
4,char_count,0.038676
5,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
7,word_count,0.035406
8,ctr,0.035156
9,scroll_rate,0.033876


Section 3 completed.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I reviewed the model's false positives and false negatives rather than relying only on the aggregate metrics.

A false positive is a page that the model ranks as positive but does not belong to the observed declining group. A false negative is an observed declining page that receives a lower model probability.

I also inspect the model's feature importance to understand which observable signals contribute most to its predictions.

These results are descriptive. Feature importance shows which signals the model used, but it does not establish that changing one of those signals will cause a page to recover. The model should therefore be treated as decision support for prioritizing review, not as causal proof.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ------------------------------------------------------------
# Error analysis
# ------------------------------------------------------------

results["error_type"] = np.select(
    [
        (
            results["is_declining_label"] == 1
        )
        &
        (
            results["model_prediction"] == 0
        ),

        (
            results["is_declining_label"] == 0
        )
        &
        (
            results["model_prediction"] == 1
        )
    ],

    [
        "False negative",
        "False positive"
    ],

    default="Correct"
)


# ------------------------------------------------------------
# Error counts
# ------------------------------------------------------------

error_counts = (
    results["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

print("Error summary:")
display(error_counts)


# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

false_positives = (
    results[
        results["error_type"]
        == "False positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
)

print("False positives:")
display(
    false_positives.head(10)
)


# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

false_negatives = (
    results[
        results["error_type"]
        == "False negative"
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
)

print("False negatives:")
display(
    false_negatives.head(10)
)


# ------------------------------------------------------------
# Highest-ranked model predictions
# ------------------------------------------------------------

top_10 = (
    results
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(10)
)

print("Top 10 model-ranked pages:")
display(top_10)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

print("Most important model features:")
display(
    feature_importance.head(10)
)

Error summary:


,error_type,count
0,Correct,1563
1,False positive,529
2,False negative,233


False positives:


,content_id,client_id,is_declining_label,model_probability,model_prediction,baseline_score,error_type
23250,content_d2dffcc697a4,client_f74efabef1,0,0.737130,1,0.570216,False positive
23559,content_00603b0349b4,client_f74efabef1,0,0.734944,1,0.410600,False positive
25913,content_331182ca4cae,client_f74efabef1,0,0.733631,1,0.450350,False positive
23750,content_e55b8ab078b0,client_f74efabef1,0,0.733059,1,0.330942,False positive
10155,content_643f585dc7f7,client_f74efabef1,0,0.731120,1,0.382338,False positive
5966,content_f5013794ba57,client_f74efabef1,0,0.730532,1,0.410767,False positive
28337,content_ea4417d89e2c,client_f74efabef1,0,0.729056,1,0.346092,False positive
4249,content_db1cd41b4b4f,client_f74efabef1,0,0.729052,1,0.674417,False positive
21530,content_b15a8dbdf66f,client_f74efabef1,0,0.727853,1,0.444824,False positive
2380,content_96da95476e63,client_f74efabef1,0,0.724807,1,0.429361,False positive


False negatives:


,content_id,client_id,is_declining_label,model_probability,model_prediction,baseline_score,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,1,0.079867,0,0.008021,False negative
3879,content_34b14c00f80c,client_d4735e3a26,1,0.082196,0,0.130394,False negative
27177,content_79ac977c6e0b,client_f74efabef1,1,0.149546,0,0.062973,False negative
22991,content_472ce7ae14c0,client_d4735e3a26,1,0.152184,0,0.147318,False negative
5608,content_a55d958ec725,client_d4735e3a26,1,0.163987,0,0.146678,False negative
12864,content_f1ef151d5e36,client_d4735e3a26,1,0.165946,0,0.146869,False negative
12076,content_230de4c50860,client_d4735e3a26,1,0.169816,0,0.146927,False negative
25838,content_cbc3b52a2ac1,client_98a3ab7c34,1,0.171345,0,0.030355,False negative
13659,content_4c437dd8c1ee,client_d4735e3a26,1,0.174066,0,0.146573,False negative
23810,content_37804210415c,client_d4735e3a26,1,0.174828,0,0.160391,False negative


Top 10 model-ranked pages:


,content_id,client_id,is_declining_label,model_probability,model_prediction,baseline_score,error_type
1085,content_0cf67ec37ab8,client_f74efabef1,1,0.768385,1,0.348221,Correct
1476,content_6e792cf3ce56,client_f74efabef1,1,0.763496,1,0.426808,Correct
1958,content_6e17dbac0491,client_f74efabef1,1,0.763129,1,0.269355,Correct
11184,content_52b1c884e871,client_f74efabef1,1,0.753808,1,0.304490,Correct
20830,content_575fd096bff5,client_f74efabef1,1,0.753051,1,0.247971,Correct
22661,content_df6b110a55c3,client_f74efabef1,1,0.750078,1,0.368645,Correct
2503,content_07fc318d9ed0,client_f74efabef1,1,0.741518,1,0.523011,Correct
1452,content_d275a7e021d8,client_f74efabef1,1,0.740485,1,0.506654,Correct
28951,content_83be0494c955,client_f74efabef1,1,0.739492,1,0.332684,Correct
6497,content_eb53dc14317a,client_f74efabef1,1,0.739256,1,0.360920,Correct


Most important model features:


,feature,importance
0,days_with_impressions,0.134951
1,log_impressions_90d,0.129377
2,avg_position,0.109203
3,content_age_days,0.092048
4,char_count,0.038676
5,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
7,word_count,0.035406
8,ctr,0.035156
9,scroll_rate,0.033876


### Interpretation

The model's errors show that the Random Forest does not perfectly separate the observed declining and non-declining groups. This is expected because the available signals do not contain every factor that affects search performance.

The feature-importance results provide a useful description of which available signals the model relied on most. The false-positive and false-negative examples also show where the model's ranking can disagree with the observed target.

The final decision should therefore remain a human-reviewed prioritization step. The model can provide additional ranking signal when it improves the baseline on the held-out test set, but it does not prove that a recommended content change will cause future performance improvement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.